# M5 Demand — Exploratory Analysis

Explore the demand-regime landscape that drives the benchmark. We look at the
distribution of mean demand, intermittency (ADI) and variability (CV²), and the
Syntetos-Boylan classification. The same plots work on the real M5 panel — just
swap `make_synthetic_panel` for `load_m5`.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.synthetic import make_synthetic_panel
from src.data.preprocessing import classify_panel
# from src.data.m5 import load_m5   # for the real dataset

panel = make_synthetic_panel(n_series=500, length=730, seed=42)
len(panel)

In [ ]:
stats = classify_panel(panel)
df = pd.DataFrame([s.__dict__ for s in stats.values()])
df[["mean", "adi", "cv2", "zero_frac", "sb_class", "volume_bucket"]].describe(include="all")

## Demand-regime landscape (ADI vs. CV²)
The Syntetos-Boylan quadrants: smooth, erratic, intermittent and lumpy.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for cls, sub in df.groupby("sb_class"):
    ax.scatter(sub["adi"], sub["cv2"], label=cls, alpha=0.6, s=18)
ax.axvline(1.32, ls="--", c="grey"); ax.axhline(0.49, ls="--", c="grey")
ax.set_xlabel("ADI (avg inter-demand interval)"); ax.set_ylabel("CV²")
ax.set_title("Syntetos-Boylan demand classification"); ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["sb_class"].value_counts().plot.bar(ax=axes[0], title="Series per SB class")
df["volume_bucket"].value_counts().plot.bar(ax=axes[1], title="Series per volume bucket")
plt.tight_layout(); plt.show()

## Example series by regime
Note the zero-inflation on the intermittent/lumpy tail — this is exactly where
point forecasts misprice inventory and a quantile metric (WQL) is needed.

In [ ]:
want = ["dense", "erratic", "intermittent", "lumpy"]
examples = {}
for ts in panel:
    p = ts.static.get("true_profile")
    if p in want and p not in examples:
        examples[p] = ts
    if len(examples) == len(want):
        break

fig, axes = plt.subplots(len(examples), 1, figsize=(11, 8), sharex=True)
for ax, (p, ts) in zip(axes, examples.items()):
    ax.plot(ts.values[-180:], lw=0.8)
    ax.set_ylabel(p)
axes[-1].set_xlabel("day (last 180)"); plt.tight_layout(); plt.show()